Author: Wiktoria Markowicz

# Regulatory Activity of DNA — Final Approach

In this notebook, I trained models to predict regulatory activity from DNA sequences. The goal was to solve two tasks:

- classification: predicting `is_active`,
- regression: predicting `rna_dna_ratio`.

My final solution is a hybrid model. I first trained a 1D CNN inspired by DeepSTARR, using one-hot encoded DNA sequences as input. The CNN has a shared feature extractor and two heads: one for the binary classification task and one for the regression task.

I also used reverse complement augmentation, because DNA regulatory motifs can appear on both strands. During evaluation, I used test-time augmentation by averaging predictions from the original sequence and its reverse complement.

To improve the model further, I extracted CNN features and combined them with handcrafted DNA features such as GC content and k-mer frequencies. I then trained LightGBM models on these features. The final predictions are obtained by combining CNN and LightGBM outputs.

In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error

In [ ]:
DATA_PATH = "dataset.tsv"
MODEL_SAVE_PATH = "best_model_deep.pth"
HISTORY_SAVE_PATH = "training_history_deep.json"

LGB_REG_SAVE_PATH = "lgb_regressor.pkl"
LGB_CLS_SAVE_PATH = "lgb_classifier.pkl"
FINAL_CONFIG_SAVE_PATH = "final_config.json"

SEED = 42
EPOCHS = 50
BATCH_SIZE = 256
LR = 3e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.2
VAL_SIZE = 0.15
PATIENCE = 10

REG_LOSS_WEIGHT = 1
AUC_WEIGHT_IN_SCORE = 0.5

USE_SCALE_REG_TARGET = True
USE_REVERSE_COMPLEMENT = True

NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

Using device: cuda


In [ ]:
# Adapter trimming is disabled.
# I tested removing constant sequence regions, but the model performed better when using the full sequence.
ADAPTER_5 = 0
ADAPTER_3 = 0
SEQ_LEN = 230

RC_MAP = str.maketrans("ACGT", "TGCA")
DNA_MAPPING = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3,
}

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## DNA sequence preprocessing

I defined helper functions for DNA-specific preprocessing. The `reverse_complement` function is used for data augmentation and test-time augmentation. The `one_hot_encode` function converts each DNA sequence into a matrix with four channels corresponding to A, C, G, and T.

Adapter trimming is included as an optional preprocessing step, but in the final version it is disabled because using the full sequence gave better validation performance.

In [ ]:
def reverse_complement(seq: str) -> str:
    return seq.upper().translate(RC_MAP)[::-1]


def trim_adapters(seq: str, adapter_5: int = ADAPTER_5, adapter_3: int = ADAPTER_3) -> str:
    seq = seq.upper()

    if adapter_3 > 0:
        return seq[adapter_5:-adapter_3]
    return seq[adapter_5:]


def one_hot_encode(seq: str, length: int = SEQ_LEN) -> np.ndarray:
    seq = seq.upper()
    ohe = np.zeros((4, length), dtype=np.float32)

    for i, base in enumerate(seq[:length]):
        idx = DNA_MAPPING.get(base)
        if idx is not None:
            ohe[idx, i] = 1.0

    return ohe

## Loading and inspecting the dataset

I load the training dataset from a TSV file and remove rows with missing values in the input sequence or target columns. I also inspect basic dataset properties, including sequence length distribution, class balance, target distribution, and duplicated sequences.

In [ ]:
df = pd.read_csv(DATA_PATH, sep="\t")
print(df.head())
print(df.shape)
print(df.columns)

df = df.dropna(subset=["sequence", "is_active", "rna_dna_ratio"]).reset_index(drop=True)

print("\nAfter dropna:", df.shape)
print(df["sequence"].str.len().describe())
print(df["is_active"].value_counts(normalize=True))
print(df["rna_dna_ratio"].describe())
print("Duplicated sequences:", df["sequence"].duplicated().sum())

                     seq_id  \
0  DNasePeakNoPromoter24818   
1  DNasePeakNoPromoter39544   
2  DNasePeakNoPromoter21529   
3  DNasePeakNoPromoter44110   
4  DNasePeakNoPromoter54524   

                                            sequence  rna_dna_ratio  is_active  
0  TCGGTTCACGCAATGTTGCATCAGTTGGGACCGTTGTTACTGACCT...          0.490          1  
1  TCGGTTCACGCAATGCGGCCTCCCAAAGTTCTAGGATTACAGGCGT...          0.129          1  
2  TCGGTTCACGCAATGAACTCTCTACCATGAGTCAAATCCAGCCCAT...         -0.776          0  
3  TCGGTTCACGCAATGACATCGAATTTCATGGTTGGCAGAGGTAAAG...         -0.668          0  
4  TCGGTTCACGCAATGACACAAACTAAGGACTGACCCGTTGTGAGAT...         -0.638          0  
(55386, 4)
Index(['seq_id', 'sequence', 'rna_dna_ratio', 'is_active'], dtype='object')

After dropna: (55386, 4)
count    55386.0
mean       230.0
std          0.0
min        230.0
25%        230.0
50%        230.0
75%        230.0
max        230.0
Name: sequence, dtype: float64
is_active
0    0.666432
1    0.333568
Name: pro

## PyTorch dataset

I created a custom PyTorch dataset for loading DNA sequences and their targets. Each sequence is optionally augmented using reverse complement transformation and then converted to one-hot encoding.

For the regression target, I optionally apply standardization using the mean and standard deviation computed only from the training set. This helps stabilize multi-task training, because the regression target can have a different scale than the binary classification loss.

In [ ]:
class DNADataset(Dataset):
    def __init__(
        self,
        sequences,
        y_cls=None,
        y_reg=None,
        augment=False,
        use_scaled_reg=False,
        reg_mean=0.0,
        reg_std=1.0,
    ):
        self.sequences = sequences
        self.y_cls = y_cls
        self.y_reg = y_reg
        self.augment = augment
        self.use_scaled_reg = use_scaled_reg
        self.reg_mean = reg_mean
        self.reg_std = reg_std

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = trim_adapters(self.sequences[idx])

        # Reverse complement augmentation is applied only during training
        if self.augment and np.random.rand() < 0.5:
            seq = reverse_complement(seq)

        # Convert DNA sequence to one-hot representation
        x = torch.from_numpy(one_hot_encode(seq)).float()

        if self.y_cls is None or self.y_reg is None:
            return x

        y_cls = torch.tensor(self.y_cls[idx], dtype=torch.float32)
        y_reg = self.y_reg[idx]

        # Standardize regression target
        if self.use_scaled_reg:
            y_reg = (y_reg - self.reg_mean) / self.reg_std

        y_reg = torch.tensor(y_reg, dtype=torch.float32)
        return x, y_cls, y_reg

## Convolutional residual block

I used a reusable 1D convolutional block as the main building component of the CNN. Each block contains a convolutional layer, batch normalization, GELU activation, and dropout.

I also added a residual skip connection. If the number of input and output channels is different, the skip path uses a `1x1` convolution to match the dimensions. This helps stabilize training and allows the model to learn deeper sequence representations.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 7,
        dilation: int = 1,
        dropout: float = 0.1,
    ):
        super().__init__()

        pad = (kernel_size + (kernel_size - 1) * (dilation - 1)) // 2

        self.conv = nn.Sequential(
            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=pad,
                dilation=dilation,
                bias=False,
            ),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.skip = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x) + self.skip(x)

## CNN architecture

I implemented a 1D CNN inspired by DeepSTARR-style models for regulatory DNA sequence prediction. The model takes one-hot encoded DNA sequences as input, where the four channels correspond to A, C, G, and T.

The architecture has three main parts:

1. A convolutional stem that extracts low-level sequence patterns.
2. A deeper convolutional tower with residual blocks and dilated convolutions. This allows the model to capture both local motifs and wider sequence context.
3. A shared fully connected representation followed by two task-specific heads:
   - a classification head for `is_active`,
   - a regression head for `rna_dna_ratio`.

This multi-task setup allows the model to learn a shared representation of regulatory sequence features while still optimizing separate outputs for classification and regression.

In [ ]:
class DeepSTARRInspired(nn.Module):
    def __init__(self, seq_len: int = SEQ_LEN, dropout: float = 0.2):
        super().__init__()

        self.seq_len = seq_len

        self.stem = nn.Sequential(
            nn.Conv1d(4, 256, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.MaxPool1d(2),
            nn.Dropout(dropout),
        )

        self.tower = nn.Sequential(
            ConvBlock(256, 256, kernel_size=7, dilation=1, dropout=dropout),
            nn.MaxPool1d(2),

            ConvBlock(256, 512, kernel_size=5, dilation=1, dropout=dropout),
            ConvBlock(512, 512, kernel_size=5, dilation=2, dropout=dropout),
            nn.MaxPool1d(2),

            ConvBlock(512, 512, kernel_size=3, dilation=1, dropout=dropout),
            ConvBlock(512, 512, kernel_size=3, dilation=4, dropout=dropout),
        )

        self.gap = nn.AdaptiveAvgPool1d(1)

        self.shared_fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.cls_head = nn.Sequential(
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

        self.reg_head = nn.Sequential(
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward_features(self, x: torch.Tensor):
        x = self.stem(x)
        x = self.tower(x)
        x = self.gap(x).squeeze(-1)
        x = self.shared_fc(x)
        return x

    def forward(self, x: torch.Tensor):
        x = self.forward_features(x)

        cls_logit = self.cls_head(x).squeeze(-1)
        reg_out = self.reg_head(x).squeeze(-1)
        return cls_logit, reg_out

## Train-validation split

I split the dataset into training and validation subsets. The split is stratified by the binary `is_active` label, so the class proportions are similar in both subsets.

The validation set is used to monitor model performance, select the best checkpoint, tune classification thresholds, and compare model variants.

In [ ]:
sequences = df["sequence"].tolist()
y_cls = df["is_active"].astype(np.float32).values
y_reg = df["rna_dna_ratio"].astype(np.float32).values

idx_train, idx_val = train_test_split(
    np.arange(len(df)),
    test_size=VAL_SIZE,
    stratify=y_cls.astype(int),
    random_state=SEED,
)

seq_train = [sequences[i] for i in idx_train]
seq_val = [sequences[i] for i in idx_val]

y_cls_train = y_cls[idx_train]
y_cls_val = y_cls[idx_val]

y_reg_train = y_reg[idx_train]
y_reg_val = y_reg[idx_val]

reg_mean = float(y_reg_train.mean())
reg_std = float(y_reg_train.std())
if reg_std == 0:
    reg_std = 1.0

## Dataset objects and data loaders

I create separate PyTorch datasets for training and validation. Reverse complement augmentation is enabled only for the training dataset, while the validation dataset is kept unchanged to provide a stable evaluation.

The regression target is standardized using statistics computed from the training set only. Data loaders are then used to batch the data during training and validation.

In [ ]:
train_ds = DNADataset(
    seq_train,
    y_cls_train,
    y_reg_train,
    augment=USE_REVERSE_COMPLEMENT,
    use_scaled_reg=USE_SCALE_REG_TARGET,
    reg_mean=reg_mean,
    reg_std=reg_std,
)

val_ds = DNADataset(
    seq_val,
    y_cls_val,
    y_reg_val,
    augment=False,
    use_scaled_reg=USE_SCALE_REG_TARGET,
    reg_mean=reg_mean,
    reg_std=reg_std,
)

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

val_dl = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

## Evaluation function

This function evaluates the CNN on a given data loader. It computes both classification and regression metrics.

For classification, I use accuracy, F1 score, and ROC AUC. For regression, I use MSE and MAE. If the regression target was standardized during training, predictions and targets are converted back to the original scale before computing regression metrics.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, criterion_cls=None, criterion_reg=None, reg_loss_weight=1.0, reg_mean=0.0, reg_std=1.0, use_scaled_reg=False):
    model.eval()

    all_logits = []
    all_reg_preds = []
    all_y_cls = []
    all_y_reg = []
    total_loss = 0.0
    n_samples = 0

    for batch in loader:
        x, y_cls_batch, y_reg_batch = batch
        x = x.to(device)
        y_cls_batch = y_cls_batch.to(device)
        y_reg_batch = y_reg_batch.to(device)

        logits, reg_out = model(x)

        if criterion_cls is not None and criterion_reg is not None:
            loss_cls = criterion_cls(logits, y_cls_batch)
            loss_reg = criterion_reg(reg_out, y_reg_batch)
            loss = loss_cls + reg_loss_weight * loss_reg
            total_loss += loss.item() * x.size(0)

        logits_np = logits.cpu().numpy()
        reg_np = reg_out.cpu().numpy()
        y_cls_np = y_cls_batch.cpu().numpy()
        y_reg_np = y_reg_batch.cpu().numpy()

        if use_scaled_reg:
            reg_np = reg_np * reg_std + reg_mean
            y_reg_np = y_reg_np * reg_std + reg_mean

        all_logits.append(logits_np)
        all_reg_preds.append(reg_np)
        all_y_cls.append(y_cls_np)
        all_y_reg.append(y_reg_np)

        n_samples += x.size(0)

    logits = np.concatenate(all_logits)
    reg_preds = np.concatenate(all_reg_preds)
    y_cls_np = np.concatenate(all_y_cls).astype(int)
    y_reg_np = np.concatenate(all_y_reg)

    probs = torch.sigmoid(torch.from_numpy(logits)).numpy()
    preds = (probs >= 0.5).astype(int)

    metrics = {
        "loss": total_loss / max(n_samples, 1),
        "accuracy": accuracy_score(y_cls_np, preds),
        "f1": f1_score(y_cls_np, preds),
        "auc": roc_auc_score(y_cls_np, probs),
        "mse": mean_squared_error(y_reg_np, reg_preds),
        "mae": mean_absolute_error(y_reg_np, reg_preds),
    }
    return metrics

## Training function

I defined a single-epoch training function for the multi-task CNN. The model predicts both outputs at the same time: the classification logit for `is_active` and the regression value for `rna_dna_ratio`.

The total loss is a weighted sum of binary cross-entropy loss for classification and mean squared error loss for regression. I also use gradient clipping to make training more stable.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion_cls, criterion_reg, device, reg_loss_weight=1.0):
    model.train()
    total_loss = 0.0
    n_samples = 0

    for x, y_cls_batch, y_reg_batch in loader:
        x = x.to(device)
        y_cls_batch = y_cls_batch.to(device)
        y_reg_batch = y_reg_batch.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits, reg_out = model(x)

        # Multi-task loss: classification loss + weighted regression loss.
        loss_cls = criterion_cls(logits, y_cls_batch)
        loss_reg = criterion_reg(reg_out, y_reg_batch)
        loss = loss_cls + reg_loss_weight * loss_reg

        loss.backward()

        # Gradient clipping improves training stability.
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * x.size(0)
        n_samples += x.size(0)

    return total_loss / max(n_samples, 1)

## Model initialization

I initialize the CNN model and move it to the available device. I also print the number of trainable parameters to document the model size.

In [ ]:
model = DeepSTARRInspired(seq_len=SEQ_LEN, dropout=DROPOUT).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params:,}")

Trainable params: 4,305,410


## Loss functions and optimizer

For the classification task, I use `BCEWithLogitsLoss`. Since the classes are imbalanced, I use `pos_weight` computed from the training set to give more weight to the positive class.

For the regression task, I use mean squared error loss. The optimizer is AdamW with weight decay, and the learning rate is reduced automatically when the validation loss stops improving.

In [ ]:
n_pos = (y_cls_train == 1).sum()
n_neg = (y_cls_train == 0).sum()
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)

criterion_cls = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion_reg = nn.MSELoss()

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=LR / 50.0,
)

print("pos_weight:", pos_weight.item())

pos_weight: 1.9978349208831787


## CNN training loop

I train the CNN for a maximum number of epochs and evaluate it on the validation set after each epoch. The best checkpoint is selected using a combined validation score based on regression MSE and classification AUC.

Lower MSE is better, while higher AUC is better, so I minimize:

`score = validation MSE - AUC_WEIGHT_IN_SCORE * validation AUC`

This score gives preference to models that perform well on both regression and classification. I also use early stopping to avoid overfitting when the validation score stops improving.

In [ ]:
best_score = None
best_epoch = -1
best_state = None
patience_counter = 0
history = []

print(f"\n{'Epoch':>5} {'TrainLoss':>10} {'ValLoss':>10} {'Acc':>8} {'F1':>8} {'AUC':>8} {'MSE':>10} {'MAE':>10} {'Time':>8}")
print("-" * 90)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss = train_one_epoch(
        model,
        train_dl,
        optimizer,
        criterion_cls,
        criterion_reg,
        DEVICE,
        reg_loss_weight=REG_LOSS_WEIGHT,
    )

    val_metrics = evaluate(
        model,
        val_dl,
        DEVICE,
        criterion_cls=criterion_cls,
        criterion_reg=criterion_reg,
        reg_loss_weight=REG_LOSS_WEIGHT,
        reg_mean=reg_mean,
        reg_std=reg_std,
        use_scaled_reg=USE_SCALE_REG_TARGET,
    )

    scheduler.step(val_metrics["loss"])

    elapsed = time.time() - t0

    # Lower score is better.
    # The score combines regression quality and classification ranking quality.
    current_score = val_metrics["mse"] - AUC_WEIGHT_IN_SCORE * val_metrics["auc"]

    print(
        f"{epoch:5d} "
        f"{train_loss:10.4f} "
        f"{val_metrics['loss']:10.4f} "
        f"{val_metrics['accuracy']:8.4f} "
        f"{val_metrics['f1']:8.4f} "
        f"{val_metrics['auc']:8.4f} "
        f"{val_metrics['mse']:10.4f} "
        f"{val_metrics['mae']:10.4f} "
        f"{elapsed:8.1f}s"
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "score": float(current_score),
        **val_metrics,
    }
    history.append(row)

    # Main checkpoint: compromise between classification and regression.
    if best_score is None or current_score < best_score:
        best_score = float(current_score)
        best_epoch = int(epoch)
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0

        checkpoint = {
            "model_state_dict": best_state,
            "model_config": {
                "seq_len": int(SEQ_LEN),
                "dropout": float(DROPOUT),
            },
            "preprocessing": {
                "adapter_5": int(ADAPTER_5),
                "adapter_3": int(ADAPTER_3),
                "trimmed_seq_len": int(SEQ_LEN),
            },
            "training_config": {
                "scale_reg_target": bool(USE_SCALE_REG_TARGET),
                "use_reverse_complement": bool(USE_REVERSE_COMPLEMENT),
            },
            "regression_stats": {
                "mean": float(reg_mean),
                "std": float(reg_std),
            },
            "best_epoch": int(best_epoch),
            "best_score": float(best_score),
            "best_metrics": {
                "accuracy": float(val_metrics["accuracy"]),
                "f1": float(val_metrics["f1"]),
                "auc": float(val_metrics["auc"]),
                "mse": float(val_metrics["mse"]),
                "mae": float(val_metrics["mae"]),
            },
        }

        torch.save(checkpoint, MODEL_SAVE_PATH)
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}. Best epoch: {best_epoch}")
        break

with open(HISTORY_SAVE_PATH, "w") as f:
    json.dump(history, f, indent=2)

print("\nTraining finished.")
print("Best epoch:", best_epoch)
print("Best score:", best_score)
print("Model saved to:", MODEL_SAVE_PATH)
print("History saved to:", HISTORY_SAVE_PATH)


Epoch  TrainLoss    ValLoss      Acc       F1      AUC        MSE        MAE     Time
------------------------------------------------------------------------------------------
    1     1.9903     1.7358   0.5113   0.5498   0.7093     0.5403     0.5903     25.1s
    2     1.6693     2.1537   0.3390   0.5022   0.7272     0.6947     0.7029     23.9s
    3     1.6091     1.7166   0.4881   0.5562   0.7613     0.5318     0.5929     24.3s
    4     1.5515     1.6302   0.7057   0.5711   0.7531     0.5055     0.5470     25.3s
    5     1.5168     1.5257   0.6799   0.6021   0.7659     0.4613     0.5264     26.4s
    6     1.4797     1.6486   0.7427   0.5078   0.7740     0.5187     0.5315     26.0s
    7     1.4445     1.7920   0.7164   0.3214   0.7653     0.5698     0.5533     25.6s
    8     1.4138     1.5180   0.7406   0.5869   0.7793     0.4640     0.5118     25.7s
    9     1.3850     1.7734   0.7168   0.2898   0.7752     0.5496     0.5507     26.0s
   10     1.3556     1.5335   0.6196   

## Prediction dataset

I define a separate dataset class for inference and feature extraction. Unlike the training dataset, this class only returns the encoded input sequence and does not include target values or augmentation.

In [ ]:
class PredDataset(Dataset):
    def __init__(self, sequences, seq_len):
        self.sequences = sequences
        self.seq_len = seq_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = trim_adapters(self.sequences[idx])
        x = torch.from_numpy(one_hot_encode(seq, length=self.seq_len)).float()
        return x

## Handcrafted DNA features

In addition to CNN features, I also compute simple handcrafted DNA features for the LightGBM models. These features include nucleotide composition, GC content, and normalized k-mer frequencies for 2-mers, 3-mers, and 4-mers.

K-mers represent short subsequences of length `k`. They can capture local motif-like patterns that may be useful for predicting regulatory activity.

In [ ]:

from itertools import product

# ===== HANDCRAFTED DNA FEATURES =====

KMER_LISTS = {
    2: ["".join(p) for p in product("ACGT", repeat=2)],
    3: ["".join(p) for p in product("ACGT", repeat=3)],
    4: ["".join(p) for p in product("ACGT", repeat=4)],
}

KMER_INDEX = {
    k: {mer: i for i, mer in enumerate(kmers)}
    for k, kmers in KMER_LISTS.items()
}


def dna_feature_matrix(sequences, seq_len=SEQ_LEN):
    """
    Handcrafted DNA features for LightGBM:
    - A/C/G/T nucleotide frequencies
    - GC content
    - normalized 2-mer, 3-mer, and 4-mer frequencies
    """
    n_basic = 5  # A, C, G, T, GC
    n_kmers = sum(len(v) for v in KMER_LISTS.values())
    X = np.zeros((len(sequences), n_basic + n_kmers), dtype=np.float32)

    for row, seq in enumerate(sequences):
        seq = trim_adapters(seq).upper()[:seq_len]
        L = max(len(seq), 1)

        count_a = seq.count("A")
        count_c = seq.count("C")
        count_g = seq.count("G")
        count_t = seq.count("T")

        X[row, 0] = count_a / L
        X[row, 1] = count_c / L
        X[row, 2] = count_g / L
        X[row, 3] = count_t / L
        X[row, 4] = (count_g + count_c) / L

        offset = n_basic

        for k, kmers in KMER_LISTS.items():
            denom = max(L - k + 1, 1)
            idx_map = KMER_INDEX[k]

            for i in range(L - k + 1):
                mer = seq[i:i + k]
                idx = idx_map.get(mer)
                if idx is not None:
                    X[row, offset + idx] += 1.0 / denom

            offset += len(kmers)

    return X

## Prediction utilities

I define helper functions for generating CNN predictions. The first function predicts outputs for the original sequences only. The second function applies test-time augmentation by predicting both the original sequence and its reverse complement, then averaging the outputs.

This makes the predictions more robust to DNA strand orientation.

In [ ]:
@torch.no_grad()
def predict_model_outputs(model, df_input, seq_len, device, batch_size=256, scale_reg=False, reg_mean=0.0, reg_std=1.0):
    sequences = df_input["sequence"].str.upper().tolist()
    ds = PredDataset(sequences, seq_len=seq_len)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    all_probs = []
    all_reg = []

    model.eval()

    for x in dl:
        x = x.to(device)
        logits, reg_out = model(x)

        probs = torch.sigmoid(logits).cpu().numpy()
        reg_pred = reg_out.cpu().numpy()

        if scale_reg:
            reg_pred = reg_pred * reg_std + reg_mean

        all_probs.extend(probs.tolist())
        all_reg.extend(reg_pred.tolist())

    return np.array(all_probs), np.array(all_reg)


@torch.no_grad()
def predict_model_outputs_tta(model, df_input, seq_len, device, batch_size=256, scale_reg=False, reg_mean=0.0, reg_std=1.0):
    seq_orig = df_input["sequence"].str.upper().tolist()
    seq_rc = [reverse_complement(s) for s in seq_orig]

    ds_orig = PredDataset(seq_orig, seq_len=seq_len)
    ds_rc = PredDataset(seq_rc, seq_len=seq_len)

    dl_orig = DataLoader(ds_orig, batch_size=batch_size, shuffle=False, num_workers=0)
    dl_rc = DataLoader(ds_rc, batch_size=batch_size, shuffle=False, num_workers=0)

    probs_orig, reg_orig = [], []
    probs_rc, reg_rc = [], []

    model.eval()

    for x in dl_orig:
        x = x.to(device)
        logits, reg_out = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        reg_pred = reg_out.cpu().numpy()
        if scale_reg:
            reg_pred = reg_pred * reg_std + reg_mean
        probs_orig.extend(probs.tolist())
        reg_orig.extend(reg_pred.tolist())

    for x in dl_rc:
        x = x.to(device)
        logits, reg_out = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        reg_pred = reg_out.cpu().numpy()
        if scale_reg:
            reg_pred = reg_pred * reg_std + reg_mean
        probs_rc.extend(probs.tolist())
        reg_rc.extend(reg_pred.tolist())

    probs_final = (np.array(probs_orig) + np.array(probs_rc)) / 2.0
    reg_final = (np.array(reg_orig) + np.array(reg_rc)) / 2.0

    return probs_final, reg_final

## Validation metrics and threshold selection

For classification, I search for the threshold that gives the best validation accuracy. The model outputs probabilities, so the final binary prediction depends on the chosen threshold.

I report accuracy, F1 score, and AUC for classification, and MSE and MAE for regression.

In [ ]:
def find_best_threshold(y_true, probs, start=0.20, stop=0.90, step=0.01):
    best_thr = 0.5
    best_acc = -1.0

    thresholds = np.arange(start, stop + 1e-9, step)
    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        acc = accuracy_score(y_true, preds)
        if acc > best_acc:
            best_acc = acc
            best_thr = float(thr)

    return best_thr, best_acc


def evaluate_predictions(y_cls_true, y_reg_true, probs, reg_pred, threshold=0.5):
    preds = (probs >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_cls_true, preds),
        "f1": f1_score(y_cls_true, preds),
        "auc": roc_auc_score(y_cls_true, probs),
        "mse": mean_squared_error(y_reg_true, reg_pred),
        "mae": mean_absolute_error(y_reg_true, reg_pred),
        "threshold": threshold,
    }
    return metrics

## CNN feature extraction

After training the CNN, I use its shared representation layer as a feature extractor. These learned feature vectors are later combined with handcrafted DNA features and used as input to LightGBM models.

In [ ]:
@torch.no_grad()
def extract_features(model, sequences, batch_size=256):
    ds = PredDataset(sequences, seq_len=SEQ_LEN)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

    feats = []

    model.eval()
    for x in dl:
        x = x.to(DEVICE)
        f = model.forward_features(x)
        feats.append(f.cpu().numpy())

    return np.vstack(feats)

## Loading the best CNN checkpoint

After training, I load the best saved CNN checkpoint. This ensures that all later validation comparisons and LightGBM feature extraction use the best model selected during training, not necessarily the model from the final epoch.

In [ ]:
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=DEVICE, weights_only=False)

best_model = DeepSTARRInspired(
    seq_len=checkpoint["model_config"]["seq_len"],
    dropout=checkpoint["model_config"]["dropout"],
).to(DEVICE)

best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

print("Loaded best model from epoch:", checkpoint["best_epoch"])
print("Best metrics:", checkpoint["best_metrics"])

Loaded best model from epoch: 18
Best metrics: {'accuracy': 0.7586663456909003, 'f1': 0.6248830682881198, 'auc': 0.8168864266484052, 'mse': 0.4042533040046692, 'mae': 0.4785618185997009}


## LightGBM models on CNN and DNA features

After training the CNN, I use it as a feature extractor. For each sequence, I extract the learned CNN representation from the shared feature layer.

I also compute features for the reverse complement sequence. This is useful because regulatory DNA activity should often be robust to strand orientation.

The final LightGBM input features combine:
- CNN features from the original sequence,
- CNN features from the reverse complement,
- handcrafted DNA features such as GC content and k-mer frequencies.

In [ ]:
from lightgbm import LGBMRegressor, LGBMClassifier
import joblib

## CNN feature extraction for LightGBM

I extract CNN embeddings for both the original validation/training sequences and their reverse complements. These embeddings are later used as tabular features for LightGBM models.

In [ ]:
print("Extracting CNN features...")

seq_train_rc = [reverse_complement(s) for s in seq_train]
seq_val_rc = [reverse_complement(s) for s in seq_val]

X_train_feats = extract_features(best_model, seq_train)
X_train_feats_rc = extract_features(best_model, seq_train_rc)

X_val_feats = extract_features(best_model, seq_val)
X_val_feats_rc = extract_features(best_model, seq_val_rc)

print("CNN features:")
print("X_train_feats:", X_train_feats.shape)
print("X_train_feats_rc:", X_train_feats_rc.shape)
print("X_val_feats:", X_val_feats.shape)
print("X_val_feats_rc:", X_val_feats_rc.shape)

Extracting CNN features...
CNN features:
X_train_feats: (47078, 256)
X_train_feats_rc: (47078, 256)
X_val_feats: (8308, 256)
X_val_feats_rc: (8308, 256)


## Adding handcrafted DNA features

I add handcrafted sequence-level features to the learned CNN features. These include nucleotide composition, GC content, and normalized 2-mer, 3-mer, and 4-mer frequencies.

This gives LightGBM access to simple motif-like sequence statistics that may not be perfectly captured by the CNN representation alone.

In [ ]:
print("\nExtracting handcrafted DNA features...")

X_train_dna = dna_feature_matrix(seq_train)
X_train_dna_rc = dna_feature_matrix(seq_train_rc)

X_val_dna = dna_feature_matrix(seq_val)
X_val_dna_rc = dna_feature_matrix(seq_val_rc)

X_train_feats = np.hstack([X_train_feats, X_train_dna])
X_train_feats_rc = np.hstack([X_train_feats_rc, X_train_dna_rc])

X_val_feats = np.hstack([X_val_feats, X_val_dna])
X_val_feats_rc = np.hstack([X_val_feats_rc, X_val_dna_rc])

print("\nAfter adding handcrafted DNA features:")
print("X_train_feats:", X_train_feats.shape)
print("X_train_feats_rc:", X_train_feats_rc.shape)
print("X_val_feats:", X_val_feats.shape)
print("X_val_feats_rc:", X_val_feats_rc.shape)

## LightGBM regressor

For the regression task, I train a LightGBM regressor on an augmented feature matrix. I include both the original sequence features and reverse complement features as separate training examples with the same regression target.

During validation, I predict the regression value for both orientations and average the two predictions.

In [ ]:
print("\nTraining LightGBM regressor...")

X_train_reg_aug = np.vstack([X_train_feats, X_train_feats_rc])
y_reg_train_aug = np.concatenate([y_reg_train, y_reg_train])

lgb_reg = LGBMRegressor(
    n_estimators=900,
    learning_rate=0.025,
    max_depth=5,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    verbosity=-1,
    force_col_wise=True,
)

lgb_reg.fit(X_train_reg_aug, y_reg_train_aug)

reg_pred_val_lgb_orig = lgb_reg.predict(X_val_feats)
reg_pred_val_lgb_rc = lgb_reg.predict(X_val_feats_rc)

reg_pred_val_lgb = 0.5 * reg_pred_val_lgb_orig + 0.5 * reg_pred_val_lgb_rc


Training LightGBM regressor...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## LightGBM classifier

For the classification task, I train a LightGBM classifier using the same augmented feature representation. I use `class_weight="balanced"` to reduce the effect of class imbalance.

As with regression, validation probabilities are averaged over the original sequence and its reverse complement.

In [27]:
print("\nTraining LightGBM classifier...")

X_train_cls_aug = np.vstack([X_train_feats, X_train_feats_rc])
y_cls_train_aug = np.concatenate([
    y_cls_train.astype(int),
    y_cls_train.astype(int),
])

lgb_cls = LGBMClassifier(
    n_estimators=900,
    learning_rate=0.025,
    max_depth=5,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=SEED,
    verbosity=-1,
    force_col_wise=True,
)

lgb_cls.fit(X_train_cls_aug, y_cls_train_aug)

probs_val_lgb_cls_orig = lgb_cls.predict_proba(X_val_feats)[:, 1]
probs_val_lgb_cls_rc = lgb_cls.predict_proba(X_val_feats_rc)[:, 1]

probs_val_lgb_cls = 0.5 * probs_val_lgb_cls_orig + 0.5 * probs_val_lgb_cls_rc


Training LightGBM classifier...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Saving LightGBM models

I save the trained LightGBM regressor and classifier so they can be loaded later by the evaluation script.

In [28]:
joblib.dump(lgb_reg, LGB_REG_SAVE_PATH)
joblib.dump(lgb_cls, LGB_CLS_SAVE_PATH)

print("\nSaved:")
print("-", LGB_REG_SAVE_PATH)
print("-", LGB_CLS_SAVE_PATH)


Saved:
- lgb_regressor.pkl
- lgb_classifier.pkl


## Validation dataframe

I reconstruct the validation dataframe using the saved validation indices. This allows me to evaluate all final prediction variants on exactly the same validation subset.

In [29]:
val_eval_df = df.iloc[idx_val].reset_index(drop=True)

y_val_cls = val_eval_df["is_active"].astype(int).values
y_val_reg = val_eval_df["rna_dna_ratio"].astype(float).values

## Validation comparison of model variants

I compare several prediction variants:

1. plain CNN predictions,
2. CNN predictions with test-time reverse complement averaging,
3. LightGBM classifier on CNN and DNA features,
4. CNN + LightGBM probability ensemble for classification,
5. LightGBM regressor,
6. CNN + LightGBM ensemble for regression.

For classification, I tune the probability threshold on the validation set. For the final model, I use the CNN TTA + LightGBM classifier mixture, because it achieved the best validation accuracy among the tested variants. For regression, I use a weighted average of the CNN and LightGBM regression outputs.

In [30]:
# ===== PLAIN CNN =====

probs_plain, reg_plain = predict_model_outputs(
    best_model,
    val_eval_df,
    seq_len=SEQ_LEN,
    device=DEVICE,
    batch_size=256,
    scale_reg=USE_SCALE_REG_TARGET,
    reg_mean=reg_mean,
    reg_std=reg_std,
)

best_thr_plain, best_acc_plain = find_best_threshold(y_val_cls, probs_plain)

metrics_plain_05 = evaluate_predictions(
    y_val_cls,
    y_val_reg,
    probs_plain,
    reg_plain,
    threshold=0.5,
)

metrics_plain_bestthr = evaluate_predictions(
    y_val_cls,
    y_val_reg,
    probs_plain,
    reg_plain,
    threshold=best_thr_plain,
)


# ===== TTA CNN =====

probs_tta, reg_tta = predict_model_outputs_tta(
    best_model,
    val_eval_df,
    seq_len=SEQ_LEN,
    device=DEVICE,
    batch_size=256,
    scale_reg=USE_SCALE_REG_TARGET,
    reg_mean=reg_mean,
    reg_std=reg_std,
)

best_thr_tta, best_acc_tta = find_best_threshold(y_val_cls, probs_tta)

metrics_tta_05 = evaluate_predictions(
    y_val_cls,
    y_val_reg,
    probs_tta,
    reg_tta,
    threshold=0.5,
)

metrics_tta_bestthr = evaluate_predictions(
    y_val_cls,
    y_val_reg,
    probs_tta,
    reg_tta,
    threshold=best_thr_tta,
)


# ===== LIGHTGBM CLASSIFIER ONLY =====

best_thr_lgb_cls, best_acc_lgb_cls = find_best_threshold(
    y_val_cls,
    probs_val_lgb_cls,
)

metrics_lgb_cls = evaluate_predictions(
    y_val_cls,
    y_val_reg,
    probs_val_lgb_cls,
    reg_tta,
    threshold=best_thr_lgb_cls,
)


# ===== MIX CLASSIFICATION: CNN TTA + LGBM CLASSIFIER =====

best_cls_alpha = None
best_cls_threshold = None
best_cls_acc = -1.0
best_probs_cls_mix = None

for alpha in np.linspace(0, 1, 101):
    probs_cls_mix = alpha * probs_tta + (1.0 - alpha) * probs_val_lgb_cls

    thr, acc = find_best_threshold(
        y_val_cls,
        probs_cls_mix,
        start=0.20,
        stop=0.90,
        step=0.01,
    )

    if acc > best_cls_acc:
        best_cls_acc = float(acc)
        best_cls_alpha = float(alpha)
        best_cls_threshold = float(thr)
        best_probs_cls_mix = probs_cls_mix.copy()

metrics_cls_mix = evaluate_predictions(
    y_val_cls,
    y_val_reg,
    best_probs_cls_mix,
    reg_tta,
    threshold=best_cls_threshold,
)


# ===== LIGHTGBM REGRESSOR ONLY =====

metrics_lgb_reg = evaluate_predictions(
    y_val_cls,
    y_val_reg,
    probs_tta,
    reg_pred_val_lgb,
    threshold=best_thr_tta,
)


# ===== MIX REGRESSION: CNN TTA + LGBM REGRESSOR =====

best_reg_alpha = None
best_reg_mse = float("inf")
best_reg_mix = None

for alpha in np.linspace(0, 1, 101):
    reg_mix_candidate = alpha * reg_tta + (1.0 - alpha) * reg_pred_val_lgb

    mse = mean_squared_error(y_val_reg, reg_mix_candidate)

    if mse < best_reg_mse:
        best_reg_mse = float(mse)
        best_reg_alpha = float(alpha)
        best_reg_mix = reg_mix_candidate.copy()

metrics_reg_mix = evaluate_predictions(
    y_val_cls,
    y_val_reg,
    probs_tta,
    best_reg_mix,
    threshold=best_thr_tta,
)


# ===== FINAL CLASSIFICATION: CNN TTA + LGBM CLASSIFIER =====

final_cls_probs = best_probs_cls_mix
final_cls_threshold = best_cls_threshold
final_cls_source = "main_cnn_lgb"


# ===== FINAL MIX: BEST CLASSIFICATION + BEST REGRESSION =====

metrics_final_mix = evaluate_predictions(
    y_val_cls,
    y_val_reg,
    final_cls_probs,
    best_reg_mix,
    threshold=final_cls_threshold,
)


# ===== COMPARISON =====

comparison = pd.DataFrame([
    {"variant": "plain_0.5", **metrics_plain_05},
    {"variant": "plain_bestthr", **metrics_plain_bestthr},
    {"variant": "tta_0.5", **metrics_tta_05},
    {"variant": "tta_bestthr", **metrics_tta_bestthr},
    {"variant": "lgb_classifier_only", **metrics_lgb_cls},
    {"variant": "cls_mix_cnn_lgb", **metrics_cls_mix},
    {"variant": "lgb_regressor_only", **metrics_lgb_reg},
    {"variant": "reg_mix_cnn_lgb", **metrics_reg_mix},
    {"variant": "final_mix_cls_and_reg", **metrics_final_mix},
])

print("Best CNN threshold:", best_thr_tta)

print("Best main CNN+LGB classification alpha:", best_cls_alpha)
print("Best main CNN+LGB classification threshold:", best_cls_threshold)
print("Best main CNN+LGB classification accuracy:", best_cls_acc)

print("Selected classification source:", final_cls_source)
print("Selected classification threshold:", final_cls_threshold)
print("Selected classification accuracy:", metrics_final_mix["accuracy"])

print("Best regression alpha:", best_reg_alpha)
print("Best regression MSE:", best_reg_mse)

comparison

Best CNN threshold: 0.5400000000000003
Best main CNN+LGB classification alpha: 0.22
Best main CNN+LGB classification threshold: 0.6400000000000003
Best main CNN+LGB classification accuracy: 0.7745546461242176
Selected classification source: main_cnn_lgb
Selected classification threshold: 0.6400000000000003
Selected classification accuracy: 0.7745546461242176
Best regression alpha: 0.17
Best regression MSE: 0.3653023406978198


,variant,accuracy,f1,auc,mse,mae,threshold
0,plain_0.5,0.758666,0.624883,0.816886,0.404253,0.478562,0.50
1,plain_bestthr,0.763722,0.598486,0.816886,0.404253,0.478562,0.57
2,tta_0.5,0.769499,0.632649,0.826016,0.400201,0.473986,0.50
3,tta_bestthr,0.773110,0.618189,0.826016,0.400201,0.473986,0.54
4,lgb_classifier_only,0.773231,0.632319,0.829435,0.400201,0.473986,0.66
5,cls_mix_cnn_lgb,0.774555,0.628299,0.829978,0.400201,0.473986,0.64
6,lgb_regressor_only,0.773110,0.618189,0.826016,0.366869,0.458638,0.54
7,reg_mix_cnn_lgb,0.773110,0.618189,0.826016,0.365302,0.456691,0.54
8,final_mix_cls_and_reg,0.774555,0.628299,0.829978,0.365302,0.456691,0.64


## Saving final configuration

I save the final configuration used by the selected model. This includes preprocessing settings, regression scaling statistics, reverse complement settings, ensemble weights, selected classification threshold, and paths to the saved model files.

The same configuration should be used by the external evaluation script.

In [31]:
# ===== SAVE FINAL CONFIG =====

final_config = {
    "seq_len": int(SEQ_LEN),
    "adapter_5": int(ADAPTER_5),
    "adapter_3": int(ADAPTER_3),
    "use_scaled_reg": bool(USE_SCALE_REG_TARGET),
    "reg_mean": float(reg_mean),
    "reg_std": float(reg_std),
    "reg_loss_weight": float(REG_LOSS_WEIGHT),

    "use_tta": True,
    "lgb_use_reverse_complement": True,
    "lgb_feature_mode": "vstack_rc_average",
    "use_handcrafted_dna_features": True,
    "kmer_features": [2, 3, 4],

    "classification": {
        "source": final_cls_source,
        "threshold": float(final_cls_threshold),
        "main_cnn_lgb_alpha": float(best_cls_alpha),
    },

    "regression": {
        "cnn_lgb_alpha": float(best_reg_alpha),
    },

    "model_paths": {
        "cnn": MODEL_SAVE_PATH,
        "lgb_classifier": LGB_CLS_SAVE_PATH,
        "lgb_regressor": LGB_REG_SAVE_PATH,
    },

    "validation_metrics": {
        "final_mix": {
            k: float(v) if isinstance(v, (np.floating, float, int, np.integer)) else v
            for k, v in metrics_final_mix.items()
        }
    }
}

with open(FINAL_CONFIG_SAVE_PATH, "w") as f:
    json.dump(final_config, f, indent=2)

print("Saved final config to:", FINAL_CONFIG_SAVE_PATH)

Saved final config to: final_config.json
